# Level 1 — Task 3: Exploratory Data Analysis (EDA)
**Internship:** Codveda Technology — Business Analytics  
**Objective:** Analyze and summarize datasets to find patterns, trends, and anomalies using statistical summaries, visualizations, and correlation/regression analysis.  
**Datasets Used:**
- `churn_cleaned.csv` — Telecom customer churn (cleaned in Task 2)
- `iris_cleaned.csv` — Iris flower dataset (cleaned in Task 2)

---

## 0. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
PALETTE = {'False': '#3498db', 'True': '#e74c3c',
           False  : '#3498db', True  : '#e74c3c'}

print('Libraries loaded ✓')

---
## 1. Load Cleaned Data
> These files were produced by `Task2_Data_Cleaning.ipynb`. Run that notebook first if the files don't exist.

In [ ]:
import os
DATA_DIR = '../data'

churn = pd.read_csv(os.path.join(DATA_DIR, 'churn_cleaned.csv'))
iris  = pd.read_csv(os.path.join(DATA_DIR, 'iris_cleaned.csv'))

# Convert Churn to readable labels for plotting
churn['Churn_label'] = churn['Churn'].map({True: 'Churned', False: 'Retained',
                                            1: 'Churned', 0: 'Retained'})

print(f'Churn : {churn.shape}')
print(f'Iris  : {iris.shape}')
display(churn.head(3))
display(iris.head(3))

---
## 2. Statistical Summaries
Descriptive statistics: mean, median, mode, standard deviation, skewness, and kurtosis.

In [ ]:
def descriptive_stats(df, name='Dataset'):
    """
    Reusable: returns extended descriptive stats including median, mode,
    skewness, and kurtosis for all numerical columns.
    """
    num_df = df.select_dtypes(include=['float64','int64'])
    desc = num_df.describe().T
    desc['median']   = num_df.median()
    desc['mode']     = num_df.mode().iloc[0]
    desc['skewness'] = num_df.skew().round(3)
    desc['kurtosis'] = num_df.kurt().round(3)
    desc = desc[['count','mean','median','mode','std','min','25%','75%','max','skewness','kurtosis']]
    desc = desc.round(3)
    print(f'── Descriptive Statistics: {name} ──')
    return desc

display(descriptive_stats(churn, 'Churn Dataset'))
display(descriptive_stats(iris,  'Iris Dataset'))

---
## 3. Target Variable Distribution — Churn
Understanding the class balance is critical — imbalanced classes affect model performance.

In [ ]:
churn_counts = churn['Churn_label'].value_counts()
churn_pct    = churn['Churn_label'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
colors = ['#3498db', '#e74c3c']
bars = axes[0].bar(churn_counts.index, churn_counts.values, color=colors,
                   edgecolor='white', width=0.5)
for bar, val, pct in zip(bars, churn_counts.values, churn_pct.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 f'{val:,}\n({pct:.1f}%)', ha='center', fontweight='bold', fontsize=11)
axes[0].set_title('Customer Churn Distribution', fontweight='bold')
axes[0].set_ylabel('Number of Customers')
axes[0].set_ylim(0, churn_counts.max() * 1.2)

# Pie chart
axes[1].pie(churn_counts.values, labels=churn_counts.index,
            colors=colors, autopct='%1.1f%%', startangle=90,
            textprops={'fontsize': 12})
axes[1].set_title('Churn Proportion', fontweight='bold')

plt.suptitle('Target Variable Analysis — Customer Churn', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('churn_distribution.png', bbox_inches='tight')
plt.show()
print(f'\nChurn rate: {churn_pct.get("Churned", churn_pct.iloc[0]):.1f}%')

---
## 4. Feature Distributions — Histograms
Histograms reveal the shape, spread, and skewness of each feature.

In [ ]:
def plot_histograms(df, columns, title='Feature Distributions', color='#3498db', filename=None):
    """Reusable: plots histogram grid for a list of columns."""
    n = len(columns)
    cols_per_row = 3
    rows = (n + cols_per_row - 1) // cols_per_row
    fig, axes = plt.subplots(rows, cols_per_row, figsize=(15, rows * 4))
    axes = axes.flatten()

    for i, col in enumerate(columns):
        axes[i].hist(df[col].dropna(), bins=35, color=color, edgecolor='white', alpha=0.85)
        mean_val   = df[col].mean()
        median_val = df[col].median()
        axes[i].axvline(mean_val,   color='#e74c3c', linestyle='--', linewidth=1.5, label=f'Mean: {mean_val:.1f}')
        axes[i].axvline(median_val, color='#2ecc71', linestyle='-',  linewidth=1.5, label=f'Median: {median_val:.1f}')
        axes[i].set_title(col, fontweight='bold', fontsize=10)
        axes[i].legend(fontsize=8)
        axes[i].set_xlabel('Value')
        axes[i].set_ylabel('Frequency')

    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle(title, fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    if filename:
        plt.savefig(filename, bbox_inches='tight')
    plt.show()

num_cols_churn = ['Account length','Total day minutes','Total day charge',
                  'Total eve minutes','Total night minutes',
                  'Total intl minutes','Customer service calls']

plot_histograms(churn, num_cols_churn,
                title='Churn Dataset — Feature Distributions',
                color='#3498db',
                filename='churn_histograms.png')

plot_histograms(iris, ['sepal_length','sepal_width','petal_length','petal_width'],
                title='Iris Dataset — Feature Distributions',
                color='#9b59b6',
                filename='iris_histograms.png')

---
## 5. Scatter Plots — Relationships Between Variables

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

scatter_pairs = [
    ('Total day minutes', 'Total day charge',   'Day Usage vs Charge'),
    ('Customer service calls', 'Total day minutes', 'Service Calls vs Day Minutes'),
    ('Account length', 'Total day minutes',     'Account Age vs Day Usage'),
]

for ax, (x, y, title) in zip(axes, scatter_pairs):
    churned   = churn[churn['Churn_label'] == 'Churned']
    retained  = churn[churn['Churn_label'] == 'Retained']
    ax.scatter(retained[x], retained[y], alpha=0.35, s=15,
               color='#3498db', label='Retained')
    ax.scatter(churned[x],  churned[y],  alpha=0.5,  s=15,
               color='#e74c3c', label='Churned')
    ax.set_xlabel(x, fontsize=9)
    ax.set_ylabel(y, fontsize=9)
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle('Scatter Analysis — Churn vs Retention Patterns', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('scatter_churn.png', bbox_inches='tight')
plt.show()

In [ ]:
# Iris pairplot — classic EDA visualization
fig = sns.pairplot(iris, hue='species', height=2.5, corner=True,
                   plot_kws={'alpha': 0.6, 's': 30},
                   palette=['#3498db','#e74c3c','#2ecc71'])
fig.figure.suptitle('Iris Dataset — Pairplot by Species', y=1.02,
                     fontsize=13, fontweight='bold')
plt.savefig('iris_pairplot.png', bbox_inches='tight')
plt.show()

---
## 6. Trends, Seasonality & Anomalies
Identifying patterns in customer behaviour grouped by churn status.

In [ ]:
# Usage pattern by churn status
usage_cols = ['Total day minutes','Total eve minutes',
              'Total night minutes','Total intl minutes']
usage_by_churn = churn.groupby('Churn_label')[usage_cols].mean()

ax = usage_by_churn.T.plot(kind='bar', figsize=(12, 5),
                            color=['#3498db','#e74c3c'],
                            edgecolor='white', width=0.6)
plt.title('Average Usage Minutes by Time-of-Day — Churned vs Retained',
          fontweight='bold', fontsize=13)
plt.xlabel('Usage Category')
plt.ylabel('Average Minutes')
plt.xticks(rotation=15)
plt.legend(title='Customer Status')

# Annotate bars
for container in ax.containers:
    ax.bar_label(container, fmt='%.0f', fontsize=8, padding=2)

plt.tight_layout()
plt.savefig('usage_trends.png', bbox_inches='tight')
plt.show()

print('\nAverage usage by churn status:')
display(usage_by_churn.round(1))

In [ ]:
# Anomaly: customer service calls vs churn rate
service_churn = churn.groupby('Customer service calls')['Churn_label']\
                     .apply(lambda x: (x == 'Churned').mean() * 100).reset_index()
service_churn.columns = ['Service Calls', 'Churn Rate (%)']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(service_churn['Service Calls'], service_churn['Churn Rate (%)'],
            color='#e74c3c', edgecolor='white', alpha=0.85)
axes[0].set_title('Churn Rate by Number of Service Calls\n⚠ Anomaly: 4+ calls = sharp churn spike',
                   fontweight='bold')
axes[0].set_xlabel('Number of Customer Service Calls')
axes[0].set_ylabel('Churn Rate (%)')
for i, row in service_churn.iterrows():
    axes[0].text(row['Service Calls'], row['Churn Rate (%)']+0.5,
                 f"{row['Churn Rate (%)']:.0f}%", ha='center', fontsize=9, fontweight='bold')

# Intl plan vs churn
intl_churn = churn.groupby('International plan')['Churn_label']\
                  .apply(lambda x: (x=='Churned').mean()*100).reset_index()
intl_churn.columns = ['International Plan', 'Churn Rate (%)']
axes[1].bar(intl_churn['International Plan'], intl_churn['Churn Rate (%)'],
            color=['#3498db','#e74c3c'], edgecolor='white', width=0.4)
axes[1].set_title('Churn Rate: International Plan\n⚠ Intl Plan holders churn significantly more',
                   fontweight='bold')
axes[1].set_xlabel('Has International Plan')
axes[1].set_ylabel('Churn Rate (%)')
for i, row in intl_churn.iterrows():
    axes[1].text(i, row['Churn Rate (%)']+0.5,
                 f"{row['Churn Rate (%)']:.1f}%", ha='center', fontsize=11, fontweight='bold')

plt.suptitle('Anomaly Detection — Churn Risk Drivers', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('anomaly_detection.png', bbox_inches='tight')
plt.show()

---
## 7. Correlation Analysis
Correlation quantifies the linear relationship between variables.  
Range: **-1 (perfect negative)** to **+1 (perfect positive)**. Values near 0 = no linear relationship.

In [ ]:
def plot_correlation_heatmap(df, columns, title='Correlation Matrix', filename=None):
    """Reusable: plots a styled correlation heatmap."""
    corr = df[columns].corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))  # upper triangle mask

    fig, ax = plt.subplots(figsize=(12, 9))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
                center=0, vmin=-1, vmax=1, square=True,
                linewidths=0.5, ax=ax,
                annot_kws={'size': 9})
    ax.set_title(title, fontweight='bold', fontsize=13)
    plt.xticks(rotation=40, ha='right', fontsize=9)
    plt.yticks(fontsize=9)
    plt.tight_layout()
    if filename:
        plt.savefig(filename, bbox_inches='tight')
    plt.show()
    return corr

churn_num_cols = ['Account length','Total day minutes','Total day calls',
                  'Total day charge','Total eve minutes','Total eve calls',
                  'Total night minutes','Total intl minutes',
                  'Customer service calls']

corr_matrix = plot_correlation_heatmap(
    churn, churn_num_cols,
    title='Correlation Heatmap — Churn Dataset',
    filename='correlation_heatmap_churn.png'
)

iris_corr = plot_correlation_heatmap(
    iris, ['sepal_length','sepal_width','petal_length','petal_width'],
    title='Correlation Heatmap — Iris Dataset',
    filename='correlation_heatmap_iris.png'
)

In [ ]:
def top_correlations(corr_matrix, n=10):
    """Reusable: returns the top N strongest correlations from a matrix."""
    pairs = (corr_matrix.where(np.tril(np.ones(corr_matrix.shape), k=-1).astype(bool))
                        .stack()
                        .reset_index())
    pairs.columns = ['Feature 1','Feature 2','Correlation']
    pairs['Abs Corr'] = pairs['Correlation'].abs()
    return pairs.sort_values('Abs Corr', ascending=False).head(n).drop('Abs Corr', axis=1)

print('── Top 10 Correlations — Churn Dataset ──')
display(top_correlations(corr_matrix))

---
## 8. Regression Analysis
Simple linear regression between Total Day Minutes and Total Day Charge to confirm the pricing model.

In [ ]:
def simple_linear_regression(df, x_col, y_col, title='', filename=None):
    """
    Reusable: fits and plots simple linear regression between two columns.
    Returns slope, intercept, and R².
    """
    x = df[x_col].values
    y = df[y_col].values

    slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
    r_squared = r_value ** 2

    x_line = np.linspace(x.min(), x.max(), 200)
    y_line = slope * x_line + intercept

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.scatter(x, y, alpha=0.25, s=12, color='#3498db', label='Data points')
    ax.plot(x_line, y_line, color='#e74c3c', linewidth=2,
            label=f'Regression line\ny = {slope:.4f}x + {intercept:.4f}')
    ax.set_xlabel(x_col, fontweight='bold')
    ax.set_ylabel(y_col, fontweight='bold')
    ax.set_title(f'{title}\nR² = {r_squared:.4f} | p-value = {p_value:.2e}',
                 fontweight='bold')
    ax.legend()
    plt.tight_layout()
    if filename:
        plt.savefig(filename, bbox_inches='tight')
    plt.show()

    print(f'  Slope     : {slope:.6f}')
    print(f'  Intercept : {intercept:.6f}')
    print(f'  R²        : {r_squared:.6f}  (explains {r_squared*100:.1f}% of variance)')
    print(f'  p-value   : {p_value:.2e}  ({"Significant" if p_value < 0.05 else "Not significant"})')
    return slope, intercept, r_squared

print('── Regression: Day Minutes → Day Charge ──')
simple_linear_regression(
    churn, 'Total day minutes', 'Total day charge',
    title='Linear Regression: Day Usage Minutes vs Charge',
    filename='regression_day_minutes.png'
)

---
## 9. Multi-Feature EDA — Iris Species Analysis
Comparing feature distributions across the three Iris species.

In [ ]:
iris_features = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
axes = axes.flatten()
colors = ['#3498db', '#e74c3c', '#2ecc71']

for ax, col in zip(axes, iris_features):
    species_list = iris['species'].unique()
    data_groups  = [iris[iris['species'] == s][col].values for s in species_list]
    bp = ax.boxplot(data_groups, patch_artist=True, labels=species_list)
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    for median in bp['medians']:
        median.set_color('black')
        median.set_linewidth(2)
    ax.set_title(col.replace('_',' ').title(), fontweight='bold')
    ax.set_ylabel('cm')

plt.suptitle('Iris Feature Distributions by Species', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('iris_species_boxplots.png', bbox_inches='tight')
plt.show()

---
## 10. Key Business Metrics & KPI Summary

In [ ]:
churn_rate = (churn['Churn_label'] == 'Churned').mean() * 100
avg_day_minutes_churned  = churn[churn['Churn_label']=='Churned']['Total day minutes'].mean()
avg_day_minutes_retained = churn[churn['Churn_label']=='Retained']['Total day minutes'].mean()
high_risk_customers = (churn['Customer service calls'] >= 4).sum()
intl_churn_rate = churn[churn['International plan']=='Yes']['Churn_label']\
                      .apply(lambda x: x=='Churned').mean() * 100

print('━'*55)
print('  📊 KEY BUSINESS METRICS — CHURN EDA SUMMARY')
print('━'*55)
print(f'  Overall Churn Rate              : {churn_rate:.1f}%')
print(f'  Avg Day Minutes (Churned)       : {avg_day_minutes_churned:.1f} min')
print(f'  Avg Day Minutes (Retained)      : {avg_day_minutes_retained:.1f} min')
print(f'  High-Risk Customers (≥4 calls)  : {high_risk_customers:,}')
print(f'  Intl Plan Churn Rate            : {intl_churn_rate:.1f}%')
print(f'  Total Customers Analysed        : {len(churn):,}')
print('━'*55)

---
## 11. Insights & Conclusions

### Churn Dataset
| Finding | Business Implication |
|---------|---------------------|
| ~14% churn rate | Significant revenue leakage |
| Churned customers use more day minutes | High usage ≠ loyalty; pricing may be the issue |
| 4+ service calls → sharp churn spike | Support quality is a key retention driver |
| International plan holders churn more | Pricing for intl plans likely uncompetitive |
| Day minutes & charge are ~perfectly correlated | Consistent per-minute pricing confirmed |

### Iris Dataset
| Finding | Implication |
|---------|-------------|
| Petal length/width best separates species | Most useful features for classification |
| Setosa is cleanly separable | Easier to classify than Versicolor/Virginica |
| Sepal width shows overlap across species | Less discriminative feature |

> **Next Step:** These insights feed directly into Level 3 — Predictive ML (churn prediction model) and Risk Analysis.